<a href="https://colab.research.google.com/github/jrebull/Vision/blob/main/CompresiondeImagenes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
!pip install pillow-heif

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 27.3 MB/s eta 0:00:00


In [7]:
import os
from pathlib import Path
from PIL import Image
import zipfile
import logging
import pillow_heif # <--- Importamos la nueva librería

# --- Activamos el lector de HEIC ---
# Esto le dice a la librería principal de imágenes "Oye, aprende a leer HEIC"
pillow_heif.register_heif_opener()
# ----------------------------------

# --- Configuración ---

# ¡¡IMPORTANTE!!
# Asegúrate de que esta ruta sea 100% correcta.
input_folder_path = Path('/content/drive/MyDrive/MNA_Vision/7.2_Characteristics_Extraction') # <--- ¡VERIFICA ESTO!

# Carpeta temporal en Colab
output_folder_path = Path('/content/imagenes_procesadas_jpg')
output_folder_path.mkdir(exist_ok=True)

# Nombre del archivo ZIP
zip_file_name = '/content/imagenes_comprimidas_Ceci.zip'

# Extensiones de imagen (incluyendo HEIC en mayúsculas y minúsculas)
image_extensions = ['.png', '.jpeg', '.jpg', '.bmp', '.tiff', '.webp', '.heic']

# --- Fin de la Configuración ---

processed_files_paths = []
logging.basicConfig(level=logging.INFO, format='%(message)s')
log = logging.getLogger()

# Verificar si la ruta de entrada existe
if not input_folder_path.exists():
    log.error(f"¡ERROR! La ruta de entrada no existe: {input_folder_path}")
    log.error("Por favor, verifica la variable 'input_folder_path' en el script.")
else:
    log.info(f"--- Iniciando proceso (con soporte HEIC activado) ---")
    log.info(f"Buscando imágenes en: {input_folder_path} (y todas sus subcarpetas)")

    # 1. Buscar recursivamente
    for image_file in input_folder_path.rglob('*'):

        # Obtenemos la extensión y la pasamos a minúsculas
        file_ext = image_file.suffix.lower()

        # Verificamos si la extensión está en nuestra lista
        if file_ext in image_extensions:
            try:
                # Crear un nombre único
                output_name = f"{image_file.parent.name}_{image_file.stem}.jpg"
                output_file_path = output_folder_path / output_name

                # Abrir la imagen (¡Ahora SÍ debería poder abrir .heic!)
                with Image.open(image_file) as img:
                    # Convertir a RGB
                    if img.mode == 'RGBA' or img.mode == 'P':
                        img = img.convert('RGB')

                    # Guardar como JPG comprimido
                    img.save(output_file_path, 'JPEG', quality=85, optimize=True)

                    processed_files_paths.append(output_file_path)
                    log.info(f"Procesado: {image_file.relative_to(input_folder_path)} -> {output_file_path.name}")

            except Exception as e:
                log.warning(f"Error al procesar {image_file.name}: {e}")


    log.info(f"\n--- {len(processed_files_paths)} imágenes convertidas exitosamente. ---")

    # 2. Crear el archivo ZIP
    if processed_files_paths:
        log.info(f"Creando archivo ZIP en: {zip_file_name}")
        try:
            with zipfile.ZipFile(zip_file_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
                for file_path in processed_files_paths:
                    zipf.write(file_path, arcname=file_path.name)

            log.info(f"¡Archivo ZIP '{zip_file_name}' creado exitosamente!")
            log.info("--- Proceso finalizado ---")
            log.info("¡Ahora puedes ejecutar la Celda 3 para descargar el archivo!")
        except Exception as e:
            log.error(f"Error al crear el ZIP: {e}")
    else:
        log.warning("--- Proceso finalizado ---")
        log.warning("No se encontraron imágenes para procesar. Verifica la ruta y las extensiones.")